In [ ]:
import fastspec, fastspec.spec as s, dataclasses, inspect
print(fastspec.__file__)
print([n for n in dir(s) if not n.startswith('_')])
print(inspect.signature(s.SpecParser.from_openapi))
print([f.name for f in dataclasses.fields(s.OpSpec)])

In [ ]:
# error: ImportError: cannot import name 'SpecParser' from 'fastspec' (/Users/71293/code/personal/orgs/shalya/.venv/lib/python3.13/site-packages/fastspec/__init__.py)
import fastspec, inspect
from fastspec import SpecParser, openapi_to_ops
print([n for n in dir(SpecParser) if not n.startswith('_')])
print(inspect.signature(openapi_to_ops))
print(inspect.getdoc(SpecParser))
spec = {'openapi':'3.0.0','info':{'title':'petstore','version':'1.0'},
 'paths':{'/pets':{'get':{'operationId':'listPets','tags':['pets'],'summary':'List pets','parameters':[{'name':'limit','in':'query','schema':{'type':'integer'}}]}},
          '/pets/{petId}':{'get':{'operationId':'getPet','tags':['pets'],'summary':'One pet','parameters':[{'name':'petId','in':'path','required':True,'schema':{'type':'string'}}]}}}}
from fastcore.basics import AttrDict
p = SpecParser.from_openapi(AttrDict(spec))
print(type(p), len(p.ops))
for o in p.ops: print(o.name, o.group, o.verb, o.path, o.summary, o.route_params, o.query_params, o.required_params)

In [ ]:
import inspect
from fastspec.spec import SpecParser, openapi_to_ops
from fastcore.basics import AttrDict
print([n for n in dir(SpecParser) if not n.startswith('_')])
print(inspect.signature(openapi_to_ops))
spec = {'openapi':'3.0.0','info':{'title':'petstore','version':'1.0'},
 'paths':{'/pets':{'get':{'operationId':'listPets','tags':['pets'],'summary':'List pets','parameters':[{'name':'limit','in':'query','schema':{'type':'integer'}}]}},
          '/pets/{petId}':{'get':{'operationId':'getPet','tags':['pets'],'summary':'One pet','parameters':[{'name':'petId','in':'path','required':True,'schema':{'type':'string'}}]}}}}
p = SpecParser.from_openapi(AttrDict(spec))
print(type(p), len(p.ops))
for o in p.ops: print(repr(o.name), o.group, o.verb, o.path, repr(o.summary), o.route_params, o.query_params, o.required_params)
print(p.base_url if hasattr(p,'base_url') else 'no base_url')

In [ ]:
import tempfile
from pathlib import Path
from vishalakshi import Vault
vp = Path(tempfile.mkdtemp())/'vault.db'
v = Vault(vp, offline=True)
d = v.note('the kettle boils at 100C', title='kettles', tags=['home'])
print('note ->', type(d), sorted(d) if isinstance(d, dict) else d)
hits = v.search('kettle')
print('search ->', type(hits), len(hits), (dict(hits[0]) if hits else None))
print('toc ->', v.toc()[:3])
print('topic_tree ->', list(v.topic_tree()))

In [ ]:
sec = v.read(d['doc_id'] + '#1')
print('read ->', type(sec), sorted(sec) if isinstance(sec, dict) else str(sec)[:200])
w = v.watch('check the kettle', action='remind', every='1d')
print('watch ->', {k: w[k] for k in ('id','action','target','every','next_run')})
ws = v.watches()
print('watches ->', len(ws), ws[0]['id'])
print('due ->', len(v.watches(due_only=True)))
res = v.poll()
print('poll ->', type(res), res if not isinstance(res, dict) else {k: res[k] for k in list(res)[:8]})
print('unwatch ->', v.unwatch(w['id']), len(v.watches()))
print('forget ->', v.forget(d['doc_id']), len(v.search('kettle')))

In [ ]:
d2 = v.note('a second note about ovens', title='ovens', tags=['home'])
print('all toc titles ->', [t['title'] for t in v.toc()])
print('narrowed ->', [t['title'] for t in v.toc(d2['doc_id'])])

In [ ]:
import json
from pathlib import Path
from fastcore.basics import AttrDict

class Specs:
    "The api backend `LocalHost` forwards to: fastspec reads the spec, httpx makes the call."
    def __init__(self): self.parsed, self.base = {}, {}
    def api_load(self, src, name=''):
        from fastspec.spec import SpecParser
        spec = src if isinstance(src, dict) else json.loads(Path(src).read_text())
        p = SpecParser.from_openapi(AttrDict(spec))
        key = name or (spec.get('info') or {}).get('title') or 'api'
        self.parsed[key] = p
        self.base[key] = ((spec.get('servers') or [{}])[0].get('url') or '').rstrip('/')
        return {'name': key, 'operations': len(p.ops), 'groups': sorted({o.group or '' for o in p.ops})}
    def _rows(self, group='', name='', match=''):
        ops = self.parsed[name].ops if name else next(iter(self.parsed.values())).ops
        return [{'name': o.name, 'group': o.group or '', 'verb': o.verb, 'path': o.path, 'summary': o.summary or ''}
                for o in ops if (not group or o.group == group) and (not match or match in o.name)]
    def api_ops(self, group='', name='', match='', limit=None, offset=0):
        rows = self._rows(group, name, match)
        return rows[offset:] if not limit else rows[offset:offset+int(limit)]
    def api_count(self, group='', name='', match=''): return len(self._rows(group, name, match))
    def api_call(self, operation, name='', **params):
        import httpx
        key = name or next(iter(self.parsed))
        op = next(o for o in self.parsed[key].ops if o.name == operation)
        path = op.path.format(**{k: params.pop(k) for k in list(params) if f'{{{k}}}' in op.path})
        r = httpx.request(op.verb, self.base[key] + path, params=params, timeout=30)
        return {'status': r.status_code, 'body': r.text[:200]}

GH = {'openapi': '3.0.0', 'info': {'title': 'github', 'version': '1.0'},
      'servers': [{'url': 'https://api.github.com'}],
      'paths': {'/zen': {'get': {'operationId': 'getZen', 'tags': ['meta'], 'summary': 'A design maxim'}},
                '/repos/{owner}/{repo}': {'get': {'operationId': 'getRepo', 'tags': ['repos'], 'summary': 'One repository',
                    'parameters': [{'name': 'owner', 'in': 'path', 'required': True, 'schema': {'type': 'string'}},
                                   {'name': 'repo', 'in': 'path', 'required': True, 'schema': {'type': 'string'}}]}}}}
sp = Specs()
print(sp.api_load(GH))
print(sp.api_ops())
print(sp.api_count(group='meta'), sp.api_ops(match='zen'))
print(sp.api_call('get_zen'))